In [1]:
import hashlib
import json
import sys
from io import BytesIO
from urllib import request

import numpy as np
import onnxruntime as ort
from PIL import Image

In [2]:
# Download the reference model, its external weights and the sample image
PREFIX = "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
!curl.exe -fsSL -o hair_classifier_v1.onnx.data "{PREFIX}/hair_classifier_v1.onnx.data"
!curl.exe -fsSL -o hair_classifier_v1.onnx "{PREFIX}/hair_classifier_v1.onnx"
!curl.exe -fsSL -o sample.jpeg "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"

In [3]:
# Verify the SHA-256 checksums against asset_manifest.json
import hashlib, json

manifest = json.load(open("asset_manifest.json"))
files = {"model": "hair_classifier_v1.onnx",
         "external_data": "hair_classifier_v1.onnx.data",
         "sample_image": "sample.jpeg"}

for key, name in files.items():
    actual = hashlib.sha256(open(name, "rb").read()).hexdigest()     # hash of the downloaded file
    expected = manifest[key]["sha256"]                               # hash from the manifest
    print(f"{name}\n  file:     {actual}\n  manifest: {expected}\n  match:    {actual == expected}")

hair_classifier_v1.onnx
  file:     2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8
  manifest: 2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8
  match:    True
hair_classifier_v1.onnx.data
  file:     6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e
  manifest: 6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e
  match:    True
sample.jpeg
  file:     6b3ae4003b5b6d54b51ee4d5fc5f183126bb1be0bf7066e5a846785fdcbfd2f5
  manifest: 6b3ae4003b5b6d54b51ee4d5fc5f183126bb1be0bf7066e5a846785fdcbfd2f5
  match:    True


In [4]:
# Install the pinned dependencies for this homework (already done)
# %pip install -r requirements.txt

##### Question 1 — look at the ONNX graph

In [5]:
# Load the model and inspect its input and output nodes
session = ort.InferenceSession("hair_classifier_v1.onnx")   # also loads the .onnx.data file next to it

for i in session.get_inputs():
    print("Input: ", i.name, i.shape, i.type)
for o in session.get_outputs():
    print("Output:", o.name, o.shape, o.type)

Input:  input ['s77', 3, 200, 200] tensor(float)
Output: output ['s77', 1] tensor(float)


**Explanation:** The model has one input `input` with shape `[batch, 3, 200, 200]` (`s77` is the
dynamic batch dimension) and one output `output` with shape `[batch, 1]`:
the probability of straight hair.

##### Image processing

In [6]:
# Image preprocessing: same as evaluation_transform in module 8, without torchvision
def download_image(url):
    with request.urlopen(url, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")       # download + force RGB


def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)         # resize (bilinear)
    array = np.asarray(image, dtype=np.float32) / 255.0                 # pixels to 0-1
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )                                                                   # normalize
    return np.transpose(array, (2, 0, 1))[None, ...]                    # (H, W, C) -> (1, C, H, W)

##### Question 2 — target size

In [7]:
# Check the target size produced by prepare_image
image = Image.open("sample.jpeg").convert("RGB")
print("Original size:", image.size)               
print("Prepared shape:", prepare_image(image).shape) 

Original size: (1024, 1024)
Prepared shape: (1, 3, 200, 200)


##### Question 3 — normalized input

In [8]:
# First value of the R channel after preprocessing
tensor = prepare_image(image)

print(round(float(tensor[0, 0, 0, 0]), 3))   # batch 0, channel 0 (R), row 0, column 0

-1.056


**Explanation:** The value is negative because of the normalization (value − mean) / std:
the top-left pixel has a red value of about 62/255, which is below the mean of 0.485.

##### Question 4 — local ONNX inference

In [9]:
# Load the ONNX model (structure + weights) to run on the CPU
session = ort.InferenceSession(
    "hair_classifier_v1.onnx",
    providers=["CPUExecutionProvider"],
)

# Run the model on the prepared image and get the predicted probability
output = session.run(
    ["output"],
    {"input": tensor},
)[0]

print(round(float(output[0, 0]), 3))

0.728


##### Lambda handler

In [10]:
# Test the Lambda handler locally (without Docker)
!{sys.executable} smoke_test.py

{'straight_probability': 0.727697, 'straight': True}


##### Container

In [12]:
# Call the Lambda container running locally on port 9000
url = "http://localhost:9000/2015-03-31/functions/function/invocations"
payload = {"image_url": manifest["sample_image"]["url"]}   # same sample image URL as in the manifest

req = request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
print(request.urlopen(req).read().decode())

{"statusCode": 200, "headers": {"Content-Type": "application/json"}, "body": "{\"straight_probability\": 0.727697, \"straight\": true}"}


##### Question 5 — look at the Lambda configuration

In [14]:
# Show the base image declared in the Dockerfile
print([line.strip() for line in open("Dockerfile") if line.startswith("FROM")])

['FROM public.ecr.aws/lambda/python:3.13@sha256:0fc65c4c9bace18ce7cb5b670119b847140a9c1c45f2b24d06ce4e35bef0a9be']


##### Question 6 — invoke the container

In [15]:
# Parse the JSON body of the Lambda response and report straight_probability
response = json.loads(request.urlopen(req).read())   # outer JSON (statusCode, headers, body)
body = json.loads(response["body"])                   # body is a JSON string itself
print(round(body["straight_probability"], 3))

0.728
